# Establishing the baseline data and model for Monitoring

In [ ]:
import json
import os
import sys

from joblib import dump

rootpath = os.path.dirname(os.getcwd())
sys.path.append(os.path.join(rootpath, "scripts"))
sys.path.append(os.path.join(rootpath, "src"))

loading gcs configs for None


In [ ]:
from gcp_functions import read_file_as_df, upload_directory
from load_configs import Configs

from data import (
    build_features,
    create_X_y_multistep,
    split_train_test_panel,
)
from models import create_fit_xgbregressor_chain, evaluate_all

configs = Configs()

In [2]:
f_ref_path = {
    "prefix": "cleaned_samples_prod",
    "fname": "Kaggle_Access_2025-07-22_WSPall_from_2020-07-22.parquet",
}

In [4]:
f_ref_full = f_ref_path["prefix"] + "/" + f_ref_path["fname"]
df_ref = read_file_as_df(
    configs.cloud["gcs"]["project"],
    configs.cloud["gcs"]["data_monitoring_bucket"],
    f_ref_full,
)

In [5]:
df = df_ref.copy()
params = {"CldrFeats": "True", "ModReg": "True", "lags": "3", "steps": "5"}
TARGET = "returns"
SAMPLE_TICKERS = ["AAPL", "AMZN"]

In [6]:
df_train, df_test = split_train_test_panel(df, train_ratio=0.8)
df_train_feats, _features2scale = build_features(
    df_train, lags=int(params["lags"]), split="train", CldrFeats=params["CldrFeats"]
)
df_test_feats, _features2scale = build_features(
    df_test, lags=int(params["lags"]), split="test", CldrFeats=params["CldrFeats"]
)
X_train, y_train = create_X_y_multistep(
    df_train_feats, steps=int(params["steps"]), target=TARGET, split="train"
)
X_test, y_test = create_X_y_multistep(
    df_test_feats, steps=int(params["steps"]), target=TARGET, split="test"
)
# Instantiate and train a model
estimator = create_fit_xgbregressor_chain(X_train, y_train, params["ModReg"])
# Evaluate the model
scores = evaluate_all(estimator, X_train, y_train, X_test, y_test, df, SAMPLE_TICKERS)

In [11]:
ref_path = os.path.join(rootpath, "ref_data_model")

os.makedirs(ref_path, exist_ok=True)

# save params as json
with open(os.path.join(ref_path, "ref_params.json"), "w") as f:
    json.dump(params, f)
# save scores as json
with open(os.path.join(ref_path, "ref_scores.json"), "w") as f:
    json.dump(scores, f)
# save model as joblib bin
with open(os.path.join(ref_path, "model.bin"), "wb") as f:
    dump(estimator, f)
# save data as parquet
df.to_parquet(os.path.join(ref_path, "ref_data.parquet"))

In [12]:
upload_directory(
    project_id=configs.cloud["gcs"]["project"],
    bucket_name=configs.cloud["gcs"]["data_monitoring_bucket"],
    folder="ref_data_model",
    local_dir=ref_path,
)

Uploaded /home/onur/WORK/DS/repos/stocks_forecasting_live/ref_data_model/ref_params.json to gs://stocks-forecasting-data-monitoring/ref_data_model/ref_params.json
Uploaded /home/onur/WORK/DS/repos/stocks_forecasting_live/ref_data_model/model.bin to gs://stocks-forecasting-data-monitoring/ref_data_model/model.bin
Uploaded /home/onur/WORK/DS/repos/stocks_forecasting_live/ref_data_model/ref_scores.json to gs://stocks-forecasting-data-monitoring/ref_data_model/ref_scores.json
Uploaded /home/onur/WORK/DS/repos/stocks_forecasting_live/ref_data_model/ref_data.parquet to gs://stocks-forecasting-data-monitoring/ref_data_model/ref_data.parquet
Successfully uploaded 4 file(s) to gs://stocks-forecasting-data-monitoring/ref_data_model
